In [2]:
!pip install requests gTTS pygame

In [3]:
import requests
from gtts import gTTS
import pygame
import time

pygame 2.6.1 (SDL 2.28.4, Python 3.11.0)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:

def get_weather(location, language="en"):

    # Step 1: Get geographical coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": location,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(
        geo_url, params=geo_params, timeout=15
    )
    geo_response.raise_for_status()
    geo_data = geo_response.json()

    if not geo_data.get("results"):
        return "Location not found. Please enter a valid city."

    place = geo_data["results"][0]

    latitude = place["latitude"]
    longitude = place["longitude"]
    city = place["name"]

    # Step 2: Fetch weather data
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m",
        "timezone": "auto"
    }

    weather_response = requests.get(
        weather_url, params=weather_params, timeout=15
    )
    weather_response.raise_for_status()
    weather_data = weather_response.json()

    current = weather_data["current"]

    # Step 3: Weather condition descriptions
    conditions = {
        0: ("Clear sky", "ఆకాశం నిర్మలంగా ఉంది"),
        1: ("Mainly clear", "ఆకాశం ఎక్కువగా నిర్మలంగా ఉంది"),
        2: ("Partly cloudy", "కొంత మేఘావృతంగా ఉంది"),
        3: ("Overcast", "ఆకాశం పూర్తిగా మేఘావృతమై ఉంది"),
        45: ("Foggy", "పొగమంచు ఉంది"),
        48: ("Foggy", "పొగమంచు ఉంది"),
        51: ("Light drizzle", "తేలికపాటి జల్లులు పడుతున్నాయి"),
        53: ("Moderate drizzle", "మోస్తరు జల్లులు పడుతున్నాయి"),
        55: ("Dense drizzle", "భారీ జల్లులు పడుతున్నాయి"),
        61: ("Slight rain", "తేలికపాటి వర్షం పడుతోంది"),
        63: ("Moderate rain", "మోస్తరు వర్షం పడుతోంది"),
        65: ("Heavy rain", "భారీ వర్షం పడుతోంది"),
        71: ("Light snowfall", "తేలికపాటి మంచు కురుస్తోంది"),
        73: ("Moderate snowfall", "మోస్తరు మంచు కురుస్తోంది"),
        75: ("Heavy snowfall", "భారీగా మంచు కురుస్తోంది"),
        80: ("Rain showers", "వర్షపు జల్లులు పడుతున్నాయి"),
        81: ("Moderate rain showers", "మోస్తరు వర్షపు జల్లులు పడుతున్నాయి"),
        82: ("Heavy rain showers", "భారీ వర్షపు జల్లులు పడుతున్నాయి"),
        95: ("Thunderstorm", "ఉరుములతో కూడిన వర్షం పడుతోంది")
    }

    code = current["weather_code"]

    english_condition, telugu_condition = conditions.get(
        code, ("Variable weather conditions", "వాతావరణ పరిస్థితులు మారుతూ ఉన్నాయి")
    )

    # Step 4: Prepare weather announcement
    temperature = current["temperature_2m"]
    feels_like = current["apparent_temperature"]
    humidity = current["relative_humidity_2m"]
    wind = current["wind_speed_10m"]

    if language == "te":

        report = (
            f"{city} నగరానికి సంబంధించిన వాతావరణ సమాచారం. "
            f"ప్రస్తుత ఉష్ణోగ్రత {temperature} డిగ్రీల సెల్సియస్. "
            f"అనుభూతి ఉష్ణోగ్రత {feels_like} డిగ్రీల సెల్సియస్. "
            f"గాలిలో తేమ {humidity} శాతం. "
            f"{telugu_condition}. "
            f"గాలి వేగం గంటకు {wind} కిలోమీటర్లు."
        )

    else:

        report = (
            f"Weather report for {city}. "
            f"Current temperature is {temperature} degree Celsius. "
            f"Feels like {feels_like} degree Celsius. "
            f"Humidity is {humidity} percent. "
            f"Weather condition is {english_condition}. "
            f"Wind speed is {wind} kilometers per hour."
        )

    return report


In [5]:
location = input("Enter your city: ")

english_report = get_weather(location, "en")
telugu_report = get_weather(location, "te")

print("\nENGLISH WEATHER REPORT")
print(english_report)

print("\nTELUGU WEATHER REPORT")
print(telugu_report)

Enter your city:  eluru



ENGLISH WEATHER REPORT
Weather report for Eluru. Current temperature is 27.0 degree Celsius. Feels like 31.6 degree Celsius. Humidity is 88 percent. Weather condition is Rain showers. Wind speed is 13.4 kilometers per hour.

TELUGU WEATHER REPORT
Eluru నగరానికి సంబంధించిన వాతావరణ సమాచారం. ప్రస్తుత ఉష్ణోగ్రత 27.0 డిగ్రీల సెల్సియస్. అనుభూతి ఉష్ణోగ్రత 31.6 డిగ్రీల సెల్సియస్. గాలిలో తేమ 88 శాతం. వర్షపు జల్లులు పడుతున్నాయి. గాలి వేగం గంటకు 13.4 కిలోమీటర్లు.


In [7]:

def text_to_speech(text, language="en"):

    # Convert text into speech
    tts = gTTS(text=text, lang=language, slow=False)

    # Save the audio file
    filename = "weather_" + language + ".mp3"
    tts.save(filename)

    # Initialize pygame mixer if necessary
    if not pygame.mixer.get_init():
        pygame.mixer.init()

    # Load and play audio
    pygame.mixer.music.load(filename)
    pygame.mixer.music.play()

    # Wait until audio finishes
    while pygame.mixer.music.get_busy():
        time.sleep(0.1)

    # Release the audio file
    pygame.mixer.music.unload()

    print("Speech completed in", language)


In [9]:
text_to_speech(
    "Welcome to the location based weather announcement system.",
    "en"
)

Speech completed in en


In [ ]:
text_to_speech(
    "నమస్కారం! వాతావరణ సమాచార వ్యవస్థకు స్వాగతం.",
    "te"
)

In [10]:

def weather_announcement():

    print("========================================")
    print(" LOCATION BASED WEATHER ANNOUNCEMENT")
    print("========================================")

    location = input("\nEnter your location: ")

    try:

        # Get English weather report
        english_report = get_weather(location, "en")

        if "Location not found" in english_report:
            print(english_report)
            return

        # Get Telugu weather report
        telugu_report = get_weather(location, "te")

        # Display English report
        print("\n----- ENGLISH WEATHER REPORT -----")
        print(english_report)

        # Display Telugu report
        print("\n----- TELUGU WEATHER REPORT -----")
        print(telugu_report)

        # Announce in English
        print("\nPlaying English announcement...")
        text_to_speech(english_report, "en")

        # Announce in Telugu
        print("\nPlaying Telugu announcement...")
        text_to_speech(telugu_report, "te")

        print("\nWeather announcement completed!")

    except requests.exceptions.RequestException as e:
        print("Weather API error:", e)

    except Exception as e:
        print("An error occurred:", e)


# Execute the complete system
weather_announcement()


 LOCATION BASED WEATHER ANNOUNCEMENT



Enter your location:  hyderabad



----- ENGLISH WEATHER REPORT -----
Weather report for Hyderabad. Current temperature is 29.2 degree Celsius. Feels like 31.8 degree Celsius. Humidity is 54 percent. Weather condition is Overcast. Wind speed is 5.2 kilometers per hour.

----- TELUGU WEATHER REPORT -----
Hyderabad నగరానికి సంబంధించిన వాతావరణ సమాచారం. ప్రస్తుత ఉష్ణోగ్రత 28.8 డిగ్రీల సెల్సియస్. అనుభూతి ఉష్ణోగ్రత 31.3 డిగ్రీల సెల్సియస్. గాలిలో తేమ 56 శాతం. ఆకాశం పూర్తిగా మేఘావృతమై ఉంది. గాలి వేగం గంటకు 6.2 కిలోమీటర్లు.

Playing English announcement...
Speech completed in en

Playing Telugu announcement...
Speech completed in te

Weather announcement completed!
